In [39]:

from datasets import load_dataset

DATASET_NAME = "ai4privacy/pii-masking-200k"

dataset = load_dataset(DATASET_NAME)

print("Available splits:", list(dataset.keys()))

for split_name, split_data in dataset.items():
    print(f"\n--- {split_name.upper()} ---")
    print("Number of rows:", len(split_data))
    print("Columns:", split_data.column_names)
    print("Features:", split_data.features)


Available splits: ['train']

--- TRAIN ---
Number of rows: 209261
Columns: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']
Features: {'source_text': Value('string'), 'target_text': Value('string'), 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}), 'span_labels': Value('string'), 'mbert_text_tokens': List(Value('string')), 'mbert_bio_labels': List(Value('string')), 'id': Value('int64'), 'language': Value('string'), 'set': Value('string')}


In [40]:

import json

train_data = dataset["train"]

for i in range(min(3, len(train_data))):
    row = train_data[i]

    print(f"\n{'=' * 60}")
    print(f"RECORD {i + 1}")
    print(f"Language: {row['language']}")
    print(f"Set: {row['set']}")

    print("\nSource text:")
    print(row["source_text"])

    print("\nTarget text:")
    print(row["target_text"])

    print("\nPrivacy mask:")
    print(json.dumps(
        row["privacy_mask"],
        ensure_ascii=False,
        indent=2
    ))

    print("\nSpan labels:")
    print(row["span_labels"])



RECORD 1
Language: en
Set: train

Source text:
A student's assessment was found on device bearing IMEI: 06-184755-866851-3. The document falls under the various topics discussed in our Optimization curriculum. Can you please collect it?

Target text:
A student's assessment was found on device bearing IMEI: [PHONEIMEI]. The document falls under the various topics discussed in our [JOBAREA] curriculum. Can you please collect it?

Privacy mask:
[
  {
    "value": "06-184755-866851-3",
    "start": 57,
    "end": 75,
    "label": "PHONEIMEI"
  },
  {
    "value": "Optimization",
    "start": 138,
    "end": 150,
    "label": "JOBAREA"
  }
]

Span labels:
[[0, 57, "O"], [57, 75, "PHONEIMEI"], [75, 138, "O"], [138, 150, "JOBAREA"], [150, 189, "O"]]

RECORD 2
Language: en
Set: train

Source text:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.

Target text:
Dear [FIRSTNA

In [41]:


from collections import Counter

# 1. Explore language distribution
language_counts = Counter(dataset["train"]["language"])

print("Language distribution:")
for language, count in language_counts.most_common():
    print(f"{language}: {count:,}")

# 2. Filter English records only
english_data = dataset["train"].filter(
    lambda row: row["language"].strip().lower() == "en"
)

print("\nTotal records:", len(dataset["train"]))
print("English records:", len(english_data))
print(
    "English percentage:",
    round(len(english_data) / len(dataset["train"]) * 100, 2),
    "%"
)


Language distribution:
fr: 61,958
de: 52,817
it: 50,985
en: 43,501

Total records: 209261
English records: 43501
English percentage: 20.79 %


In [42]:

from collections import Counter

# Count entities and records for each label
entity_counts = Counter()
record_counts = Counter()

for row in english_data:
    labels_in_record = set()

    for entity in row["privacy_mask"]:
        label = entity["label"]

        entity_counts[label] += 1
        labels_in_record.add(label)

    record_counts.update(labels_in_record)

# Create a sorted summary
labels_summary = sorted(
    entity_counts.keys(),
    key=lambda label: entity_counts[label],
    reverse=True
)

print("Number of unique labels:", len(labels_summary))
print("\nLabel Distribution:\n")

print(f"{'Label':<25} {'Entities':>12} {'Records':>12}")
print("-" * 51)

for label in labels_summary:
    print(
        f"{label:<25} "
        f"{entity_counts[label]:>12,} "
        f"{record_counts[label]:>12,}"
    )

print("-" * 51)
print(f"{'TOTAL':<25} {sum(entity_counts.values()):>12,}")


Number of unique labels: 56

Label Distribution:

Label                         Entities      Records
---------------------------------------------------
FIRSTNAME                       13,323       12,896
LASTNAME                         4,817        4,741
DATE                             4,555        4,440
EMAIL                            4,092        4,043
PREFIX                           3,446        3,415
AMOUNT                           3,103        3,091
USERNAME                         3,056        2,984
JOBTITLE                         2,954        2,939
URL                              2,910        2,898
TIME                             2,883        2,853
JOBAREA                          2,882        2,858
MIDDLENAME                       2,881        2,871
ACCOUNTNUMBER                    2,831        2,806
STREET                           2,799        2,787
ACCOUNTNAME                      2,793        2,751
CITY                             2,787        2,718
COUNTY        

In [43]:

import pandas as pd

labels_df = pd.DataFrame([
    {
        "label": label,
        "entities": entity_counts[label],
        "records": record_counts[label],
        "record_coverage_pct": round(
            record_counts[label] / len(english_data) * 100, 2
        )
    }
    for label in entity_counts
])

labels_df = labels_df.sort_values(
    "entities", ascending=False
).reset_index(drop=True)

print("Total unique labels:", len(labels_df))
print(labels_df.to_string(index=False))


Total unique labels: 56
              label  entities  records  record_coverage_pct
          FIRSTNAME     13323    12896                29.65
           LASTNAME      4817     4741                10.90
               DATE      4555     4440                10.21
              EMAIL      4092     4043                 9.29
             PREFIX      3446     3415                 7.85
             AMOUNT      3103     3091                 7.11
           USERNAME      3056     2984                 6.86
           JOBTITLE      2954     2939                 6.76
                URL      2910     2898                 6.66
               TIME      2883     2853                 6.56
            JOBAREA      2882     2858                 6.57
         MIDDLENAME      2881     2871                 6.60
      ACCOUNTNUMBER      2831     2806                 6.45
             STREET      2799     2787                 6.41
        ACCOUNTNAME      2793     2751                 6.32
               C

In [44]:

import pandas as pd
from collections import Counter


target_labels = [
    "PASSWORD",
    "USERNAME",
    "EMAIL",
    "PHONENUMBER",
    "FIRSTNAME",
    "LASTNAME",
    "SSN",
    "CREDITCARDNUMBER",
    "CREDITCARDCVV",
    "IBAN",
    "ACCOUNTNUMBER",
    "STREET",
    "IPV4",
]


entity_counts = Counter()
record_counts = Counter()

for row in english_data:
    labels_in_record = set()

    for entity in row["privacy_mask"]:
        label = entity["label"]

        if label in target_labels:
            entity_counts[label] += 1
            labels_in_record.add(label)

    record_counts.update(labels_in_record)

summary = pd.DataFrame([
    {
        "label": label,
        "entities": entity_counts[label],
        "records": record_counts[label],
        "coverage_pct": round(
            record_counts[label] / len(english_data) * 100, 2
        ),
    }
    for label in target_labels
])

print("Labels not found:",
      [label for label in target_labels if entity_counts[label] == 0])

print("\nTotal target entities:", summary["entities"].sum())
print("Records containing at least one target label:",
      sum(
          1 for row in english_data
          if any(e["label"] in target_labels
                 for e in row["privacy_mask"])
      ))

display(summary)


Labels not found: []

Total target entities: 46146
Records containing at least one target label: 30102


,label,entities,records,coverage_pct
0,PASSWORD,2622,2607,5.99
1,USERNAME,3056,2984,6.86
2,EMAIL,4092,4043,9.29
3,PHONENUMBER,2425,2417,5.56
4,FIRSTNAME,13323,12896,29.65
5,LASTNAME,4817,4741,10.90
6,SSN,2045,2032,4.67
7,CREDITCARDNUMBER,2574,2566,5.90
8,CREDITCARDCVV,869,869,2.00
9,IBAN,1973,1968,4.52


In [45]:

# Keep records containing at least one target entity.
target_set = set(target_labels)

scoped_data = english_data.filter(
    lambda row: any(
        entity["label"] in target_set
        for entity in row["privacy_mask"]
    )
)

print("English records:", len(english_data))
print("Scoped records:", len(scoped_data))
print("Target labels:", len(target_labels))

# Inspect one record
sample = scoped_data[0]

print("\nSource text:")
print(sample["source_text"])

print("\nPrivacy mask:")
print(sample["privacy_mask"])


English records: 43501
Scoped records: 30102
Target labels: 13

Source text:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.

Privacy mask:
[{'value': 'Omer', 'start': 5, 'end': 9, 'label': 'FIRSTNAME'}, {'value': '78B5R2MVFAHJ48500', 'start': 44, 'end': 61, 'label': 'VEHICLEVIN'}]


In [46]:

target_set = set(target_labels)

records_with_both = 0
records_with_only_target = 0

for row in scoped_data:
    labels = [e["label"] for e in row["privacy_mask"]]

    has_target = any(label in target_set for label in labels)
    has_other = any(label not in target_set for label in labels)

    if has_target and has_other:
        records_with_both += 1
    else:
        records_with_only_target += 1

print("Records with target + other labels:", records_with_both)
print("Records with target labels only:", records_with_only_target)


Records with target + other labels: 27533
Records with target labels only: 2569


In [47]:

sample = scoped_data[0]

print("Source text:")
print(sample["source_text"])

print("\nTokens:")
print(sample["mbert_text_tokens"][:40])

print("\nBIO labels:")
print(sample["mbert_bio_labels"][:40])

print("\nSpan labels:")
print(sample["span_labels"])


Source text:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.

Tokens:
['Dear', 'Omer', ',', 'as', 'per', 'our', 'records', ',', 'your', 'license', '78', '##B', '##5', '##R', '##2', '##M', '##V', '##FA', '##H', '##J', '##48', '##50', '##0', 'is', 'still', 'registered', 'in', 'our', 'records', 'for', 'access', 'to', 'the', 'educational', 'tools', '.', 'Please', 'feedback', 'on', 'it']

BIO labels:
['O', 'B-FIRSTNAME', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Span labels:
[[0, 5, "O"], [5, 9, "FIRSTNAME"], [9, 44, "O"], [44, 61, "VEHICLEVIN"], [61, 170, "O"]]


In [48]:

sample = scoped_data[0]

for entity in sample["privacy_mask"]:
    extracted = sample["source_text"][
        entity["start"]:entity["end"]
    ]

    print("Label:", entity["label"])
    print("Expected:", entity["value"])
    print("Extracted:", extracted)
    print("Match:", extracted == entity["value"])
    print("-" * 30)


Label: FIRSTNAME
Expected: Omer
Extracted: Omer
Match: True
------------------------------
Label: VEHICLEVIN
Expected: 78B5R2MVFAHJ48500
Extracted: 78B5R2MVFAHJ48500
Match: True
------------------------------


In [49]:

from collections import Counter

total_entities = 0
matched_entities = 0
mismatched_entities = 0
mismatch_examples = []

for row_idx, row in enumerate(english_data):
    text = row["source_text"]

    for entity in row["privacy_mask"]:
        total_entities += 1

        start = entity["start"]
        end = entity["end"]
        expected = entity["value"]
        extracted = text[start:end]

        if extracted == expected:
            matched_entities += 1
        else:
            mismatched_entities += 1

            if len(mismatch_examples) < 10:
                mismatch_examples.append({
                    "row_index": row_idx,
                    "label": entity["label"],
                    "expected": expected,
                    "extracted": extracted,
                    "start": start,
                    "end": end,
                })

print("Total entities:", total_entities)
print("Matched entities:", matched_entities)
print("Mismatched entities:", mismatched_entities)

if total_entities:
    print(
        "Match rate:",
        round(matched_entities / total_entities * 100, 2),
        "%"
    )

print("\nMismatch examples:")
for example in mismatch_examples:
    print(example)


Total entities: 137093
Matched entities: 137093
Mismatched entities: 0
Match rate: 100.0 %

Mismatch examples:


In [50]:

target_set = set(target_labels)

target_entity_count = sum(
    1
    for row in english_data
    for entity in row["privacy_mask"]
    if entity["label"] in target_set
)

print("All English entities:", total_entities)
print("Target entities:", target_entity_count)
print("Non-target entities:", total_entities - target_entity_count)


All English entities: 137093
Target entities: 46146
Non-target entities: 90947


In [51]:

# Target labels selected for Confidentiality Detector
target_labels = [
    "PASSWORD",
    "USERNAME",
    "EMAIL",
    "PHONENUMBER",
    "FIRSTNAME",
    "LASTNAME",
    "SSN",
    "CREDITCARDNUMBER",
    "CREDITCARDCVV",
    "IBAN",
    "ACCOUNTNUMBER",
    "STREET",
    "IPV4",
]

target_set = set(target_labels)

def build_target_spans(row):
    text = row["source_text"]
    target_spans = []

    for entity in row["privacy_mask"]:
        if entity["label"] not in target_set:
            continue

        start = entity["start"]
        end = entity["end"]
        value = entity["value"]

        # Verify annotation offsets before keeping the entity
        if text[start:end] != value:
            raise ValueError(
                f"Offset mismatch: {entity['label']} | {value!r}"
            )

        target_spans.append({
            "start": start,
            "end": end,
            "label": entity["label"],
            "value": value,
        })

    return target_spans

# Create a new column without modifying the original annotations
scoped_data = scoped_data.map(
    lambda row: {"target_spans": build_target_spans(row)}
)

print("Scoped records:", len(scoped_data))
print("Target labels:", len(target_labels))
print("Example target spans:")
print(scoped_data[0]["target_spans"])


Scoped records: 30102
Target labels: 13
Example target spans:
[{'start': 5, 'end': 9, 'label': 'FIRSTNAME', 'value': 'Omer'}]


In [52]:

sample = scoped_data[0]

print("Source text:")
print(sample["source_text"])

print("\nOriginal labels:")
print(sorted(set(
    entity["label"] for entity in sample["privacy_mask"]
)))

print("\nTarget labels kept:")
print(sorted(set(
    entity["label"] for entity in sample["target_spans"]
)))

print("\nNon-target labels excluded:")
print(sorted(set(
    entity["label"] for entity in sample["privacy_mask"]
    if entity["label"] not in target_set
)))


Source text:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.

Original labels:
['FIRSTNAME', 'VEHICLEVIN']

Target labels kept:
['FIRSTNAME']

Non-target labels excluded:
['VEHICLEVIN']


In [53]:

print("Number of records:", len(scoped_data))
print("Available columns:", scoped_data.column_names)

print("\nTarget labels:", target_labels)

print("\nSample record:")
print(scoped_data[0]["source_text"])


Number of records: 30102
Available columns: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set', 'target_spans']

Target labels: ['PASSWORD', 'USERNAME', 'EMAIL', 'PHONENUMBER', 'FIRSTNAME', 'LASTNAME', 'SSN', 'CREDITCARDNUMBER', 'CREDITCARDCVV', 'IBAN', 'ACCOUNTNUMBER', 'STREET', 'IPV4']

Sample record:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.


In [54]:

# Inspect target spans in the first record
sample = scoped_data[0]

print("Text:")
print(sample["source_text"])

print("\nTarget spans:")
for span in sample["target_spans"]:
    print(span)

print("\nValidation:")
for span in sample["target_spans"]:
    extracted_text = sample["source_text"][span["start"]:span["end"]]
    print(
        f"Label: {span['label']} | "
        f"Value: {span['value']} | "
        f"Extracted: {extracted_text} | "
        f"Match: {extracted_text == span['value']}"
    )


Text:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.

Target spans:
{'start': 5, 'end': 9, 'label': 'FIRSTNAME', 'value': 'Omer'}

Validation:
Label: FIRSTNAME | Value: Omer | Extracted: Omer | Match: True


In [55]:

from transformers import AutoTokenizer

model_checkpoint = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print("Tokenizer loaded successfully!")
print("Tokenizer:", model_checkpoint)


Tokenizer loaded successfully!
Tokenizer: bert-base-multilingual-cased


In [56]:

text = "Dear Omer, please contact omer@example.com"

encoding = tokenizer(
    text,
    return_offsets_mapping=True
)

tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
offsets = encoding["offset_mapping"]

for token, offset in zip(tokens, offsets):
    start, end = offset
    print(f"Token: {token:20} | Offset: {offset} | Text: {text[start:end]}")


Token: [CLS]                | Offset: (0, 0) | Text: 
Token: Dear                 | Offset: (0, 4) | Text: Dear
Token: Omer                 | Offset: (5, 9) | Text: Omer
Token: ,                    | Offset: (9, 10) | Text: ,
Token: pl                   | Offset: (11, 13) | Text: pl
Token: ##eas                | Offset: (13, 16) | Text: eas
Token: ##e                  | Offset: (16, 17) | Text: e
Token: contact              | Offset: (18, 25) | Text: contact
Token: om                   | Offset: (26, 28) | Text: om
Token: ##er                 | Offset: (28, 30) | Text: er
Token: @                    | Offset: (30, 31) | Text: @
Token: example              | Offset: (31, 38) | Text: example
Token: .                    | Offset: (38, 39) | Text: .
Token: com                  | Offset: (39, 42) | Text: com
Token: [SEP]                | Offset: (0, 0) | Text: 


In [57]:

sample = scoped_data[0]
text = sample["source_text"]

encoding = tokenizer(
    text,
    return_offsets_mapping=True,
    truncation=True,
    max_length=256
)

tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
offsets = encoding["offset_mapping"]

for token, (start, end) in zip(tokens, offsets):
    if start == end:
        continue

    matched_spans = [
        span for span in sample["target_spans"]
        if start < span["end"] and end > span["start"]
    ]

    print(
        f"{token:20} | {(start, end)} | "
        f"{text[start:end]!r} | "
        f"{[span['label'] for span in matched_spans]}"
    )


Dear                 | (0, 4) | 'Dear' | []
Omer                 | (5, 9) | 'Omer' | ['FIRSTNAME']
,                    | (9, 10) | ',' | []
as                   | (11, 13) | 'as' | []
per                  | (14, 17) | 'per' | []
our                  | (18, 21) | 'our' | []
records              | (22, 29) | 'records' | []
,                    | (29, 30) | ',' | []
your                 | (31, 35) | 'your' | []
license              | (36, 43) | 'license' | []
78                   | (44, 46) | '78' | []
##B                  | (46, 47) | 'B' | []
##5                  | (47, 48) | '5' | []
##R                  | (48, 49) | 'R' | []
##2                  | (49, 50) | '2' | []
##M                  | (50, 51) | 'M' | []
##V                  | (51, 52) | 'V' | []
##FA                 | (52, 54) | 'FA' | []
##H                  | (54, 55) | 'H' | []
##J                  | (55, 56) | 'J' | []
##48                 | (56, 58) | '48' | []
##50                 | (58, 60) | '50' | []
##0               

In [58]:

IGNORE_INDEX = -100

def create_bio_labels(text, privacy_mask, offsets):
    labels = []

    for start, end in offsets:
        # Special tokens مثل [CLS] و[SEP]
        if start == end:
            labels.append(IGNORE_INDEX)
            continue

        overlapping_entities = [
            entity for entity in privacy_mask
            if start < entity["end"] and end > entity["start"]
        ]

        if not overlapping_entities:
            labels.append("O")
            continue

        # إذا تداخل الـ Token مع كيان غير مستهدف، نتجاهله
        non_target_overlap = any(
            entity["label"] not in target_set
            for entity in overlapping_entities
        )

        if non_target_overlap:
            labels.append(IGNORE_INDEX)
            continue

        # الكيان المستهدف الذي يتداخل مع هذا الـ Token
        entity = overlapping_entities[0]
        label = entity["label"]

        if start <= entity["start"] < end:
            labels.append(f"B-{label}")
        else:
            labels.append(f"I-{label}")

    return labels


sample = scoped_data[0]

sample_encoding = tokenizer(
    sample["source_text"],
    return_offsets_mapping=True,
    truncation=True,
    max_length=256
)

sample_labels = create_bio_labels(
    sample["source_text"],
    sample["privacy_mask"],
    sample_encoding["offset_mapping"]
)

sample_tokens = tokenizer.convert_ids_to_tokens(
    sample_encoding["input_ids"]
)

for token, label in zip(sample_tokens, sample_labels):
    print(f"{token:20} | {label}")


[CLS]                | -100
Dear                 | O
Omer                 | B-FIRSTNAME
,                    | O
as                   | O
per                  | O
our                  | O
records              | O
,                    | O
your                 | O
license              | O
78                   | -100
##B                  | -100
##5                  | -100
##R                  | -100
##2                  | -100
##M                  | -100
##V                  | -100
##FA                 | -100
##H                  | -100
##J                  | -100
##48                 | -100
##50                 | -100
##0                  | -100
is                   | O
still                | O
registered           | O
in                   | O
our                  | O
records              | O
for                  | O
access               | O
to                   | O
the                  | O
educational          | O
tools                | O
.                    | O
Please               | 

In [59]:

from collections import Counter

print("Label distribution in the sample:")
for label, count in Counter(sample_labels).items():
    print(f"{label}: {count}")

print("\nNumber of tokens:", len(sample_tokens))
print("Number of labels:", len(sample_labels))
print("Lengths match:", len(sample_tokens) == len(sample_labels))


Label distribution in the sample:
-100: 15
O: 31
B-FIRSTNAME: 1

Number of tokens: 47
Number of labels: 47
Lengths match: True


In [60]:

# Find a record with a target entity split across multiple tokens
found = False

for i in range(min(100, len(scoped_data))):
    row = scoped_data[i]

    encoding = tokenizer(
        row["source_text"],
        return_offsets_mapping=True,
        truncation=True,
        max_length=256
    )

    offsets = encoding["offset_mapping"]

    for entity in row["target_spans"]:
        overlapping_tokens = [
            (start, end)
            for start, end in offsets
            if start < end
            and start < entity["end"]
            and end > entity["start"]
        ]

        if len(overlapping_tokens) > 1:
            print("Record index:", i)
            print("Entity:", entity)
            print("Entity text:", row["source_text"][entity["start"]:entity["end"]])
            print("Overlapping token offsets:", overlapping_tokens)
            found = True
            break

    if found:
        break

if not found:
    print("No multi-token target entity found in the first 100 records.")


Record index: 1
Entity: {'start': 0, 'end': 6, 'label': 'FIRSTNAME', 'value': 'Kattie'}
Entity text: Kattie
Overlapping token offsets: [(0, 3), (3, 6)]


In [61]:

row = scoped_data[1]
text = row["source_text"]

encoding = tokenizer(
    text,
    return_offsets_mapping=True,
    truncation=True,
    max_length=256
)

tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])

labels = create_bio_labels(
    text,
    row["privacy_mask"],
    encoding["offset_mapping"]
)

for token, (start, end), label in zip(
    tokens,
    encoding["offset_mapping"],
    labels
):
    if start < end and start < 6 and end > 0:
        print(f"Token: {token:15} | Text: {text[start:end]!r} | Label: {label}")


Token: Kat             | Text: 'Kat' | Label: B-FIRSTNAME
Token: ##tie           | Text: 'tie' | Label: I-FIRSTNAME


In [62]:

checked_records = 0
length_mismatches = 0
missing_target_labels = 0

for i in range(min(100, len(scoped_data))):
    row = scoped_data[i]

    encoding = tokenizer(
        row["source_text"],
        return_offsets_mapping=True,
        truncation=True,
        max_length=256
    )

    labels = create_bio_labels(
        row["source_text"],
        row["privacy_mask"],
        encoding["offset_mapping"]
    )

    if len(encoding["input_ids"]) != len(labels):
        length_mismatches += 1

    offsets = encoding["offset_mapping"]

    for entity in row["target_spans"]:
        # Is the entire entity represented by the tokenized text?
        covered = any(
            start <= entity["start"] and end >= entity["end"]
            for start, end in offsets
            if start < end
        )

        if not covered:
            # A multi-token entity can still be covered by several tokens.
            covered = (
                any(start == entity["start"] for start, end in offsets)
                and any(end == entity["end"] for start, end in offsets)
            )

        if not covered:
            missing_target_labels += 1

    checked_records += 1

print("Records checked:", checked_records)
print("Token/label length mismatches:", length_mismatches)
print("Target entities not fully covered:", missing_target_labels)


Records checked: 100
Token/label length mismatches: 0
Target entities not fully covered: 0


In [63]:
# %% [A] Data quality: nulls, empties, splits, duplicates, templates, lengths
import json, re, unicodedata
import pandas as pd
from collections import Counter
 
df = english_data.to_pandas()
 
print("Nulls per column:\n", df.isna().sum().to_string())
print("Empty/whitespace source_text:", (df["source_text"].str.strip() == "").sum())
print("Records with empty privacy_mask (PII-free):", (df["privacy_mask"].apply(len) == 0).sum())
print("'set' values (full dataset):", Counter(dataset["train"]["set"]))
print("Duplicate ids:", df["id"].duplicated().sum())
 
print("\nExact duplicate source_text:", df["source_text"].duplicated().sum())
norm = df["source_text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
print("Duplicates after case/whitespace normalisation:", norm.duplicated().sum())
 
# Near-duplicates: records that become identical once PII is replaced by placeholders.
tmpl_counts = df["target_text"].value_counts()
print("Unique templates (target_text):", len(tmpl_counts), "of", len(df))
print("Records sharing their template with >=1 other record:",
      (df["target_text"].map(tmpl_counts) > 1).sum())
print("Largest template groups:\n", tmpl_counts.head(5).to_string())
 
df["n_chars"] = df["source_text"].str.len()
df["n_tokens"] = [len(ids) for ids in tokenizer(df["source_text"].tolist())["input_ids"]]
print("\n", df[["n_chars", "n_tokens"]].describe(percentiles=[.5, .9, .99]).round(1).to_string())
print("Records > 256 tokens (truncated by current max_length):", (df["n_tokens"] > 256).sum())
print("Records < 5 tokens:", (df["n_tokens"] < 5).sum())

Nulls per column:
 source_text          0
target_text          0
privacy_mask         0
span_labels          0
mbert_text_tokens    0
mbert_bio_labels     0
id                   0
language             0
set                  0
Empty/whitespace source_text: 0
Records with empty privacy_mask (PII-free): 0
'set' values (full dataset): Counter({'train': 209261})
Duplicate ids: 0

Exact duplicate source_text: 0
Duplicates after case/whitespace normalisation: 0
Unique templates (target_text): 43496 of 43501
Records sharing their template with >=1 other record: 10
Largest template groups:
 target_text
Field trip to [ZIPCODE] park is next week! Any final payments can be made to [BITCOINADDRESS].                                                                                                                                             2
Could you help me with the application for the [JOBTITLE] role at [URL]?                                                                                          

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (618 > 512). Running this sequence through the model will result in indexing errors



        n_chars  n_tokens
count  43501.0   43501.0
mean     172.8      51.1
std       65.1      22.0
min       31.0      11.0
50%      165.0      47.0
90%      247.0      78.0
99%      340.0     118.0
max     2794.0     785.0
Records > 256 tokens (truncated by current max_length): 12
Records < 5 tokens: 0


In [64]:
issues, examples = Counter(), {}
 
def note(kind, idx, info):
    issues[kind] += 1
    bucket = examples.setdefault(kind, [])
    if len(bucket) < 3:
        bucket.append((idx, info))
 
for idx, row in enumerate(english_data):
    text = row["source_text"]
    ents = sorted(row["privacy_mask"], key=lambda e: (e["start"], e["end"]))
 
    for e in ents:
        if not (0 <= e["start"] < e["end"] <= len(text)):
            note("bad_bounds", idx, e)
        if e["value"] != e["value"].strip():
            note("whitespace_padded_value", idx, e)
 
    for a, b in zip(ents, ents[1:]):
        if b["start"] < a["end"]:
            note("overlapping_entities", idx, (a["label"], a["value"], b["label"], b["value"]))
        elif b["start"] == a["end"]:
            note("touching_entities_no_gap", idx, (a["label"], a["value"], b["label"], b["value"]))
 
    # span_labels vs privacy_mask
    spans = [tuple(s) for s in json.loads(row["span_labels"]) if s[2] != "O"]
    if spans != [(e["start"], e["end"], e["label"]) for e in ents]:
        note("span_labels_mismatch", idx, spans[:3])
 
    # target_text vs source_text + privacy_mask
    parts, prev = [], 0
    for e in ents:
        parts += [text[prev:e["start"]], f"[{e['label']}]"]
        prev = e["end"]
    parts.append(text[prev:])
    if "".join(parts) != row["target_text"]:
        note("target_text_mismatch", idx, row["target_text"][:100])
 
    # Unicode hygiene
    if unicodedata.normalize("NFC", text) != text:
        note("not_NFC", idx, text[:80])
    if re.search(r"[​-‏‪-‮⁠﻿]", text):
        note("zero_width_or_bidi_chars", idx, text[:80])
if "\u00A0" in text:
    note("non_breaking_space", idx, text[:80])
 
print("Annotation issues (English):")
for k in ["bad_bounds", "whitespace_padded_value", "overlapping_entities",
          "touching_entities_no_gap", "span_labels_mismatch", "target_text_mismatch",
          "not_NFC", "zero_width_or_bidi_chars", "non_breaking_space"]:
    print(f"  {k:28} {issues[k]:>7,}")
for k, ex in examples.items():
    print(f"\n{k}:")
    for item in ex:
        print("  ", item)
 

Annotation issues (English):
  bad_bounds                         0
  whitespace_padded_value            0
  overlapping_entities               0
  touching_entities_no_gap       2,539
  span_labels_mismatch               0
  target_text_mismatch               0
  not_NFC                            0
  zero_width_or_bidi_chars           0
  non_breaking_space                 0

touching_entities_no_gap:
   (13, ('PREFIX', 'Ms.', 'MIDDLENAME', 'North'))
   (26, ('PREFIX', 'Mr.', 'LASTNAME', 'Kerluke'))
   (28, ('CURRENCYSYMBOL', '£', 'AMOUNT', '0.001b'))


In [65]:
tok_issues, tok_examples = Counter(), {}
 
def tnote(kind, idx, info):
    tok_issues[kind] += 1
    bucket = tok_examples.setdefault(kind, [])
    if len(bucket) < 5:
        bucket.append((idx, info))
 
for idx, row in enumerate(scoped_data):
    text = row["source_text"]
    enc = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=256)
    offsets = enc["offset_mapping"]
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    real = [(i, s, t) for i, (s, t) in enumerate(offsets) if s < t]
 
    for e in row["target_spans"]:
        hit = [(i, s, t) for i, s, t in real if s < e["end"] and t > e["start"]]
        if not hit:
            tnote("target_lost_to_truncation", idx, (e["label"], e["value"]))
            continue
        if hit[0][1] != e["start"] or hit[-1][2] != e["end"]:
            tnote("token_boundary_misaligned", idx,
                  (e["label"], e["value"], text[hit[0][1]:hit[-1][2]]))
        if any(tokens[i] == "[UNK]" for i, _, _ in hit):
            tnote("unk_inside_target", idx, (e["label"], e["value"]))
 
    # tokens touching more than one entity (create_bio_labels silently takes [0])
    for i, s, t in real:
        n = sum(1 for e in row["privacy_mask"] if s < e["end"] and t > e["start"])
        if n > 1:
            tnote("token_spans_multiple_entities", idx, text[s:t])
 
    # BIO validity: I-X must follow B-X or I-X (ignoring nothing; -100 breaks the chain)
    labels = create_bio_labels(text, row["privacy_mask"], offsets)
    prev = "O"
    for i, lab in enumerate(labels):
        if offsets[i][0] == offsets[i][1]:
            continue
        if isinstance(lab, str) and lab.startswith("I-") and prev not in (f"B-{lab[2:]}", lab):
            tnote("invalid_I_tag", idx, (tokens[i], lab, prev))
        prev = lab if isinstance(lab, str) else "IGN"
 
print("Scoped records checked:", len(scoped_data))
for k in ["target_lost_to_truncation", "token_boundary_misaligned", "unk_inside_target",
          "token_spans_multiple_entities", "invalid_I_tag"]:
    print(f"  {k:32} {tok_issues[k]:>7,}")
for k, ex in tok_examples.items():
    print(f"\n{k}:")
    for item in ex:
        print("  ", item)
 

Scoped records checked: 30102
  target_lost_to_truncation             53
  token_boundary_misaligned             13
  unk_inside_target                      1
  token_spans_multiple_entities        364
  invalid_I_tag                          3

token_spans_multiple_entities:
   (33, '23₩')
   (51, 'Dollar﷼868k')
   (79, '₡0')
   (98, '₨488964')
   (416, '₨5809964058878157')

target_lost_to_truncation:
   (1464, ('ACCOUNTNUMBER', '88514759'))
   (1464, ('PHONENUMBER', '08.43-82 94-44'))
   (1464, ('EMAIL', 'Brenda_Trantow96@hotmail.com'))
   (1464, ('LASTNAME', 'Marvin'))
   (3436, ('EMAIL', 'Irma_Ortiz@yahoo.com'))

token_boundary_misaligned:
   (1901, ('CREDITCARDNUMBER', '8235367392538515', '82353673925385150'))
   (1901, ('CREDITCARDCVV', '026', '5026'))
   (2666, ('PASSWORD', 'dzjCX7Numw7b', 'dzjCX7Numw7bs'))
   (6452, ('IPV4', '30.32.68.100', '30.32.68.1007'))
   (9105, ('CREDITCARDNUMBER', '9979132607620762', '99791326076207621'))

invalid_I_tag:
   (1901, ('##26', 'I-CREDITCARD

In [66]:

idx = 1464
row = scoped_data[idx]
text = row["source_text"]

enc = tokenizer(
    text,
    return_offsets_mapping=True,
    truncation=True,
    max_length=256
)

offsets = enc["offset_mapping"]

print("Last visible character:", max(end for start, end in offsets))
print("\nTarget entity coverage:")

for entity in row["target_spans"]:
    start, end = entity["start"], entity["end"]

    overlapping = [
        (s, e) for s, e in offsets
        if s < e and s < end and e > start
    ]

    fully_covered = (
        bool(overlapping)
        and min(s for s, e in overlapping) == start
        and max(e for s, e in overlapping) == end
    )

    print(
        entity["label"],
        repr(entity["value"]),
        "| Fully covered:", fully_covered,
        "| Token offsets:", overlapping
    )


Last visible character: 1154

Target entity coverage:
FIRSTNAME 'Nikolas' | Fully covered: True | Token offsets: [(10, 16), (16, 17)]
ACCOUNTNUMBER '88514759' | Fully covered: False | Token offsets: []
PHONENUMBER '08.43-82 94-44' | Fully covered: False | Token offsets: []
EMAIL 'Brenda_Trantow96@hotmail.com' | Fully covered: False | Token offsets: []
LASTNAME 'Marvin' | Fully covered: False | Token offsets: []


In [67]:

truncated_entities = []
boundary_issues = []

for idx, row in enumerate(scoped_data):
    text = row["source_text"]

    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=256
    )

    offsets = [
        (start, end)
        for start, end in enc["offset_mapping"]
        if start < end
    ]

    for entity in row["target_spans"]:
        start, end = entity["start"], entity["end"]

        overlapping = [
            (s, e) for s, e in offsets
            if s < end and e > start
        ]

        fully_covered = (
            bool(overlapping)
            and min(s for s, e in overlapping) == start
            and max(e for s, e in overlapping) == end
        )

        if not overlapping:
            truncated_entities.append((idx, entity["label"], entity["value"]))
        elif not fully_covered:
            boundary_issues.append((idx, entity["label"], entity["value"], overlapping))

print("Target entities with no overlapping token:", len(truncated_entities))
print("Target entities with partial/misaligned coverage:", len(boundary_issues))

print("\nExamples of entities with no overlapping token:")
for item in truncated_entities[:5]:
    print(item)

print("\nExamples of partial/misaligned coverage:")
for item in boundary_issues[:5]:
    print(item)


Target entities with no overlapping token: 53
Target entities with partial/misaligned coverage: 13

Examples of entities with no overlapping token:
(1464, 'ACCOUNTNUMBER', '88514759')
(1464, 'PHONENUMBER', '08.43-82 94-44')
(1464, 'EMAIL', 'Brenda_Trantow96@hotmail.com')
(1464, 'LASTNAME', 'Marvin')
(3436, 'EMAIL', 'Irma_Ortiz@yahoo.com')

Examples of partial/misaligned coverage:
(1901, 'CREDITCARDNUMBER', '8235367392538515', [(81, 84), (84, 86), (86, 87), (87, 88), (88, 89), (89, 90), (90, 92), (92, 94), (94, 96), (96, 98)])
(1901, 'CREDITCARDCVV', '026', [(96, 98), (98, 100)])
(2666, 'PASSWORD', 'dzjCX7Numw7b', [(197, 198), (198, 199), (199, 200), (200, 201), (201, 202), (202, 203), (203, 204), (204, 206), (206, 207), (207, 208), (208, 210)])
(6452, 'IPV4', '30.32.68.100', [(75, 77), (77, 78), (78, 80), (80, 81), (81, 83), (83, 84), (84, 88)])
(9105, 'CREDITCARDNUMBER', '9979132607620762', [(71, 74), (74, 76), (76, 78), (78, 80), (80, 81), (81, 82), (82, 84), (84, 85), (85, 86), (86,

In [68]:

def count_uncovered_entities(max_length):
    uncovered = 0

    for row in scoped_data:
        enc = tokenizer(
            row["source_text"],
            return_offsets_mapping=True,
            truncation=True,
            max_length=max_length
        )

        offsets = [
            (start, end)
            for start, end in enc["offset_mapping"]
            if start < end
        ]

        for entity in row["target_spans"]:
            start, end = entity["start"], entity["end"]

            has_overlap = any(
                s < end and e > start
                for s, e in offsets
            )

            if not has_overlap:
                uncovered += 1

    return uncovered


print("Uncovered entities at 256:", count_uncovered_entities(256))
print("Uncovered entities at 512:", count_uncovered_entities(512))


Uncovered entities at 256: 53
Uncovered entities at 512: 13


In [69]:

uncovered_at_512 = []

for idx, row in enumerate(scoped_data):
    enc = tokenizer(
        row["source_text"],
        return_offsets_mapping=True,
        truncation=True,
        max_length=512
    )

    offsets = [
        (start, end)
        for start, end in enc["offset_mapping"]
        if start < end
    ]

    for entity in row["target_spans"]:
        start, end = entity["start"], entity["end"]

        if not any(s < end and e > start for s, e in offsets):
            last_end = max((e for _, e in offsets), default=0)

            uncovered_at_512.append({
                "record_index": idx,
                "label": entity["label"],
                "value": entity["value"],
                "entity_start": start,
                "entity_end": end,
                "last_token_end": last_end,
                "likely_truncated": start >= last_end
            })

print("Uncovered entities at 512:", len(uncovered_at_512))

for item in uncovered_at_512[:15]:
    print(item)


Uncovered entities at 512: 13
{'record_index': 3436, 'label': 'CREDITCARDNUMBER', 'value': '0086751260351631', 'entity_start': 1810, 'entity_end': 1826, 'last_token_end': 1781, 'likely_truncated': True}
{'record_index': 3436, 'label': 'PHONENUMBER', 'value': '+67-440.763 7137', 'entity_start': 1919, 'entity_end': 1935, 'last_token_end': 1781, 'likely_truncated': True}
{'record_index': 4803, 'label': 'STREET', 'value': 'Myah Mount', 'entity_start': 1920, 'entity_end': 1930, 'last_token_end': 1795, 'likely_truncated': True}
{'record_index': 4803, 'label': 'EMAIL', 'value': 'Henry.Parisian@hotmail.com', 'entity_start': 2079, 'entity_end': 2105, 'last_token_end': 1795, 'likely_truncated': True}
{'record_index': 4803, 'label': 'USERNAME', 'value': 'Myrtis_McKenzie', 'entity_start': 2125, 'entity_end': 2140, 'last_token_end': 1795, 'likely_truncated': True}
{'record_index': 4803, 'label': 'FIRSTNAME', 'value': 'Dorris', 'entity_start': 2377, 'entity_end': 2383, 'last_token_end': 1795, 'likel

In [72]:
# Test overlapping chunks on one long record

record_index = 3436
row = scoped_data[record_index]
text = row["source_text"]
entities = row["privacy_mask"]

encoded = tokenizer(
    text,
    truncation=True,
    max_length=512,
    stride=64,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

print("Original text length:", len(text))
print("Number of chunks:", len(encoded["input_ids"]))
print("Chunk lengths:", [len(ids) for ids in encoded["input_ids"]])

for entity in entities:
    if entity["label"] not in target_set:
        continue

    covered = any(
        any(
            start <= entity["start"]
            and end >= entity["end"]
            and start != end
            for start, end in chunk_offsets
        )
        for chunk_offsets in encoded["offset_mapping"]
    )

    if not covered:
        print(
            "Still not fully covered:",
            entity["label"],
            repr(text[entity["start"]:entity["end"]]),
        )

Original text length: 2140
Number of chunks: 2
Chunk lengths: [512, 71]
Still not fully covered: LASTNAME 'Shanahan'
Still not fully covered: ACCOUNTNUMBER '54122073'
Still not fully covered: EMAIL 'Irma_Ortiz@yahoo.com'
Still not fully covered: SSN '661 58 3186'
Still not fully covered: CREDITCARDNUMBER '0086751260351631'
Still not fully covered: PHONENUMBER '+67-440.763 7137'


In [73]:
# Inspect token coverage for the reported entities

for entity in entities:
    if entity["label"] not in target_set:
        continue

    start = entity["start"]
    end = entity["end"]

    print(f"\n{entity['label']}: {text[start:end]!r}")
    print(f"Character span: ({start}, {end})")

    for chunk_idx, offsets in enumerate(encoded["offset_mapping"]):
        overlapping = [
            (s, e, text[s:e])
            for s, e in offsets
            if s < e and s < end and e > start
        ]

        if overlapping:
            print(f"  Chunk {chunk_idx}: {overlapping}")


FIRSTNAME: 'Amalia'
Character span: (9, 15)
  Chunk 0: [(9, 15, 'Amalia')]

LASTNAME: 'Shanahan'
Character span: (16, 24)
  Chunk 0: [(16, 20, 'Shan'), (20, 24, 'ahan')]

ACCOUNTNUMBER: '54122073'
Character span: (139, 147)
  Chunk 0: [(139, 142, '541'), (142, 144, '22'), (144, 145, '0'), (145, 146, '7'), (146, 147, '3')]

EMAIL: 'Irma_Ortiz@yahoo.com'
Character span: (1685, 1705)
  Chunk 0: [(1685, 1689, 'Irma'), (1689, 1690, '_'), (1690, 1695, 'Ortiz'), (1695, 1696, '@'), (1696, 1698, 'ya'), (1698, 1700, 'ho'), (1700, 1701, 'o'), (1701, 1702, '.'), (1702, 1705, 'com')]
  Chunk 1: [(1685, 1689, 'Irma'), (1689, 1690, '_'), (1690, 1695, 'Ortiz'), (1695, 1696, '@'), (1696, 1698, 'ya'), (1698, 1700, 'ho'), (1700, 1701, 'o'), (1701, 1702, '.'), (1702, 1705, 'com')]

SSN: '661 58 3186'
Character span: (1754, 1765)
  Chunk 0: [(1754, 1757, '661'), (1758, 1760, '58'), (1761, 1764, '318'), (1764, 1765, '6')]
  Chunk 1: [(1754, 1757, '661'), (1758, 1760, '58'), (1761, 1764, '318'), (1764, 1765

In [74]:
# Inspect chunk boundaries

for i, offsets in enumerate(encoded["offset_mapping"]):
    valid_offsets = [(s, e) for s, e in offsets if s < e]

    if valid_offsets:
        first_start = valid_offsets[0][0]
        last_end = valid_offsets[-1][1]

        print(f"\nChunk {i}")
        print("Character range:", first_start, "to", last_end)
        print("End of chunk text:", repr(text[max(0, last_end - 100):last_end]))

print("\nMissing entities:")
for entity in entities:
    if entity["label"] not in target_set:
        continue

    start, end = entity["start"], entity["end"]

    covered = any(
        any(s <= start and e >= end for s, e in offsets if s < e)
        for offsets in encoded["offset_mapping"]
    )

    if not covered:
        print(entity["label"], repr(text[start:end]), (start, end))


Chunk 0
Character range: 0 to 1781
End of chunk text: 'ing Irma_Ortiz@yahoo.com, 51-257688-484981-1 and sensitive ones like the 661 58 3186, 776f:75cc:bcba'

Chunk 1
Character range: 1549 to 1786
End of chunk text: 'rma_Ortiz@yahoo.com, 51-257688-484981-1 and sensitive ones like the 661 58 3186, 776f:75cc:bcba:8d61'

Missing entities:
LASTNAME 'Shanahan' (16, 24)
ACCOUNTNUMBER '54122073' (139, 147)
EMAIL 'Irma_Ortiz@yahoo.com' (1685, 1705)
SSN '661 58 3186' (1754, 1765)
CREDITCARDNUMBER '0086751260351631' (1810, 1826)
PHONENUMBER '+67-440.763 7137' (1919, 1935)


In [75]:
print("Tokenizer:", tokenizer.name_or_path)
print("Fast tokenizer:", tokenizer.is_fast)
print("Model max length:", tokenizer.model_max_length)

print("\nEncoded keys:")
print(encoded.keys())

print("\nOverflow mapping:")
print(encoded.get("overflow_to_sample_mapping"))

print("\nNumber of input chunks:", len(encoded["input_ids"]))
print("Number of offset chunks:", len(encoded["offset_mapping"]))

Tokenizer: bert-base-multilingual-cased
Fast tokenizer: True
Model max length: 512

Encoded keys:
KeysView({'input_ids': [[101, 51555, 12916, 119, 79695, 43974, 26019, 117, 146, 50725, 10531, 79515, 31478, 13028, 11206, 10111, 10106, 15198, 17004, 119, 146, 10392, 17637, 10106, 21457, 10114, 20442, 12404, 97097, 94619, 69087, 49172, 71793, 10929, 11305, 10884, 10160, 97033, 14553, 10269, 118, 47108, 119, 10747, 10124, 10978, 169, 19037, 57939, 10110, 69794, 33295, 10105, 52703, 10108, 171, 10908, 14971, 12717, 48532, 19777, 45853, 75703, 105042, 10107, 10106, 94413, 117, 11164, 38426, 119, 10882, 17067, 10106, 10105, 13533, 10230, 13216, 104911, 11069, 10457, 120, 123, 41226, 10135, 125, 120, 10193, 117, 11155, 10529, 10590, 10751, 56729, 14499, 10106, 11192, 10189, 60600, 11408, 10105, 80592, 13903, 18309, 10111, 17531, 105042, 27413, 119, 13252, 28160, 48515, 10379, 10114, 17446, 49260, 10189, 34307, 16924, 10454, 10106, 57824, 61893, 10111, 12898, 13351, 61778, 10146, 11206, 119, 10

In [76]:
encoded_test = tokenizer(
    text,
    add_special_tokens=False,
    truncation=True,
    max_length=510,
    stride=64,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

print("Original text length:", len(text))
print("Number of chunks:", len(encoded_test["input_ids"]))

for i, offsets in enumerate(encoded_test["offset_mapping"]):
    valid = [(s, e) for s, e in offsets if s < e]

    if valid:
        print(
            f"Chunk {i}: "
            f"characters {valid[0][0]}–{valid[-1][1]}, "
            f"tokens={len(encoded_test['input_ids'][i])}"
        )

Original text length: 2140
Number of chunks: 1
Chunk 0: characters 0–1781, tokens=510


In [77]:
# Manual token-window test
# No changes to the original dataset

raw = tokenizer(
    text,
    add_special_tokens=False,
    return_offsets_mapping=True,
    truncation=False,
)

token_ids = raw["input_ids"]
offsets = raw["offset_mapping"]

max_tokens = 510
stride = 64
step = max_tokens - stride

chunks = []

for start_idx in range(0, len(token_ids), step):
    end_idx = min(start_idx + max_tokens, len(token_ids))

    chunk_offsets = offsets[start_idx:end_idx]
    valid_offsets = [(s, e) for s, e in chunk_offsets if s < e]

    if valid_offsets:
        chunks.append({
            "token_start": start_idx,
            "token_end": end_idx,
            "char_start": valid_offsets[0][0],
            "char_end": valid_offsets[-1][1],
            "token_count": end_idx - start_idx,
        })

    if end_idx == len(token_ids):
        break

print("Total tokens:", len(token_ids))
print("Original text length:", len(text))
print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk}")

print("\nLast covered character:", max(c["char_end"] for c in chunks))

Total tokens: 616
Original text length: 2140
Number of chunks: 2
Chunk 0: {'token_start': 0, 'token_end': 510, 'char_start': 0, 'char_end': 1781, 'token_count': 510}
Chunk 1: {'token_start': 446, 'token_end': 616, 'char_start': 1549, 'char_end': 2140, 'token_count': 170}

Last covered character: 2140


In [78]:
# Check target entity coverage across manual chunks

for entity in entities:
    if entity["label"] not in target_set:
        continue

    start = entity["start"]
    end = entity["end"]

    covered = any(
        chunk["char_start"] <= start
        and chunk["char_end"] >= end
        for chunk in chunks
    )

    if not covered:
        print(
            "Not covered:",
            entity["label"],
            repr(text[start:end]),
            (start, end),
        )

print("Coverage check completed.")

Coverage check completed.


In [79]:
# Inspect BIO labels for the first manual chunk

first_chunk = chunks[0]

chunk_token_ids = token_ids[
    first_chunk["token_start"]:first_chunk["token_end"]
]

chunk_offsets = offsets[
    first_chunk["token_start"]:first_chunk["token_end"]
]

chunk_labels = create_bio_labels(
    text,
    entities,
    chunk_offsets
)

print("Token count:", len(chunk_token_ids))
print("Label count:", len(chunk_labels))
print("Lengths match:", len(chunk_token_ids) == len(chunk_labels))

print("\nSample tokens and labels:")
tokens = tokenizer.convert_ids_to_tokens(chunk_token_ids)

for token, offset, label in zip(tokens, chunk_offsets, chunk_labels):
    if label != "O" and label != IGNORE_INDEX:
        print(
            f"Token: {token:15} | "
            f"Offset: {offset} | "
            f"Label: {label}"
        )

Token count: 510
Label count: 510
Lengths match: True

Sample tokens and labels:
Token: Amalia          | Offset: (9, 15) | Label: B-FIRSTNAME
Token: Shan            | Offset: (16, 20) | Label: B-LASTNAME
Token: ##ahan          | Offset: (20, 24) | Label: I-LASTNAME
Token: 541             | Offset: (139, 142) | Label: B-ACCOUNTNUMBER
Token: ##22            | Offset: (142, 144) | Label: I-ACCOUNTNUMBER
Token: ##0             | Offset: (144, 145) | Label: I-ACCOUNTNUMBER
Token: ##7             | Offset: (145, 146) | Label: I-ACCOUNTNUMBER
Token: ##3             | Offset: (146, 147) | Label: I-ACCOUNTNUMBER
Token: Irma            | Offset: (1685, 1689) | Label: B-EMAIL
Token: _               | Offset: (1689, 1690) | Label: I-EMAIL
Token: Ortiz           | Offset: (1690, 1695) | Label: I-EMAIL
Token: @               | Offset: (1695, 1696) | Label: I-EMAIL
Token: ya              | Offset: (1696, 1698) | Label: I-EMAIL
Token: ##ho            | Offset: (1698, 1700) | Label: I-EMAIL
Token: ##o

In [80]:
# Inspect BIO labels for the second manual chunk

second_chunk = chunks[1]

chunk_token_ids_2 = token_ids[
    second_chunk["token_start"]:second_chunk["token_end"]
]

chunk_offsets_2 = offsets[
    second_chunk["token_start"]:second_chunk["token_end"]
]

chunk_labels_2 = create_bio_labels(
    text,
    entities,
    chunk_offsets_2
)

tokens_2 = tokenizer.convert_ids_to_tokens(chunk_token_ids_2)

print("Token count:", len(chunk_token_ids_2))
print("Label count:", len(chunk_labels_2))
print("Lengths match:", len(chunk_token_ids_2) == len(chunk_labels_2))

print("\nSensitive tokens in chunk 2:")

for token, offset, label in zip(tokens_2, chunk_offsets_2, chunk_labels_2):
    if label != "O" and label != IGNORE_INDEX:
        print(f"{token:15} | {offset} | {label}")

Token count: 170
Label count: 170
Lengths match: True

Sensitive tokens in chunk 2:
Irma            | (1685, 1689) | B-EMAIL
_               | (1689, 1690) | I-EMAIL
Ortiz           | (1690, 1695) | I-EMAIL
@               | (1695, 1696) | I-EMAIL
ya              | (1696, 1698) | I-EMAIL
##ho            | (1698, 1700) | I-EMAIL
##o             | (1700, 1701) | I-EMAIL
.               | (1701, 1702) | I-EMAIL
com             | (1702, 1705) | I-EMAIL
661             | (1754, 1757) | B-SSN
58              | (1758, 1760) | I-SSN
318             | (1761, 1764) | I-SSN
##6             | (1764, 1765) | I-SSN
008             | (1810, 1813) | B-CREDITCARDNUMBER
##6             | (1813, 1814) | I-CREDITCARDNUMBER
##7             | (1814, 1815) | I-CREDITCARDNUMBER
##51            | (1815, 1817) | I-CREDITCARDNUMBER
##26            | (1817, 1819) | I-CREDITCARDNUMBER
##0             | (1819, 1820) | I-CREDITCARDNUMBER
##35            | (1820, 1822) | I-CREDITCARDNUMBER
##16            | (1822, 18

In [81]:
# Check tokens that cross sensitive-entity boundaries

for chunk_index, chunk in enumerate(chunks):
    chunk_offsets = offsets[
        chunk["token_start"]:chunk["token_end"]
    ]

    print(f"\nChunk {chunk_index}")

    for entity in entities:
        if entity["label"] not in target_set:
            continue

        entity_start = entity["start"]
        entity_end = entity["end"]

        for token_start, token_end in chunk_offsets:
            if token_start == token_end:
                continue

            overlaps = (
                token_start < entity_end
                and token_end > entity_start
            )

            crosses_boundary = (
                overlaps
                and (
                    token_start < entity_start
                    or token_end > entity_end
                )
            )

            if crosses_boundary:
                print(
                    "Boundary overlap:",
                    entity["label"],
                    repr(text[entity_start:entity_end]),
                    "| Token:",
                    repr(text[token_start:token_end]),
                    "| Token span:",
                    (token_start, token_end),
                )


Chunk 0

Chunk 1


In [82]:
boundary_errors = []

for record_index, row in enumerate(scoped_data):
    text = row["source_text"]
    entities = row["privacy_mask"]

    raw = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )

    offsets = raw["offset_mapping"]

    for entity in entities:
        if entity["label"] not in target_set:
            continue

        entity_start = entity["start"]
        entity_end = entity["end"]

        for token_start, token_end in offsets:
            if token_start == token_end:
                continue

            overlaps = (
                token_start < entity_end
                and token_end > entity_start
            )

            crosses_boundary = (
                overlaps
                and (
                    token_start < entity_start
                    or token_end > entity_end
                )
            )

            if crosses_boundary:
                boundary_errors.append({
                    "record_index": record_index,
                    "label": entity["label"],
                    "value": text[entity_start:entity_end],
                    "token": text[token_start:token_end],
                })

print("Boundary errors:", len(boundary_errors))
print("Examples:", boundary_errors[:10])

Boundary errors: 13
Examples: [{'record_index': 1901, 'label': 'CREDITCARDNUMBER', 'value': '8235367392538515', 'token': '50'}, {'record_index': 1901, 'label': 'CREDITCARDCVV', 'value': '026', 'token': '50'}, {'record_index': 2666, 'label': 'PASSWORD', 'value': 'dzjCX7Numw7b', 'token': 'bs'}, {'record_index': 6452, 'label': 'IPV4', 'value': '30.32.68.100', 'token': '1007'}, {'record_index': 9105, 'label': 'CREDITCARDNUMBER', 'value': '9979132607620762', 'token': '21'}, {'record_index': 9105, 'label': 'CREDITCARDCVV', 'value': '147', 'token': '21'}, {'record_index': 11287, 'label': 'USERNAME', 'value': 'Wilson.OReilly', 'token': 'lys'}, {'record_index': 11404, 'label': 'LASTNAME', 'value': 'Treutel', 'token': 'ls'}, {'record_index': 16604, 'label': 'FIRSTNAME', 'value': 'Ruthie', 'token': 'Ruthie´s'}, {'record_index': 17335, 'label': 'SSN', 'value': '756.2177.3265', 'token': '53'}]


In [83]:
for error in boundary_errors:
    idx = error["record_index"]
    row = scoped_data[idx]

    text = row["source_text"]
    entities = row["privacy_mask"]

    entity = next(
        e for e in entities
        if e["label"] == error["label"]
        and text[e["start"]:e["end"]] == error["value"]
    )

    start = entity["start"]
    end = entity["end"]

    print("\n" + "=" * 60)
    print("Record:", idx)
    print("Label:", error["label"])
    print("Entity:", repr(error["value"]))
    print("Context:", repr(text[max(0, start - 30):min(len(text), end + 30)]))
    print("Entity span:", (start, end))


Record: 1901
Label: CREDITCARDNUMBER
Entity: '8235367392538515'
Context: 'ls for Auto Loan Account with 8235367392538515026 as old details have expire'
Entity span: (81, 97)

Record: 1901
Label: CREDITCARDCVV
Entity: '026'
Context: ' Account with 8235367392538515026 as old details have expired.'
Entity span: (97, 100)

Record: 2666
Label: PASSWORD
Entity: 'dzjCX7Numw7b'
Context: ' about the dangers of sharing dzjCX7Numw7bs online.'
Entity span: (197, 209)

Record: 6452
Label: IPV4
Entity: '30.32.68.100'
Context: 'riginated from the IP address 30.32.68.100736b:bc4f:5df3:3eee:e69b:fbd7:'
Entity span: (75, 87)

Record: 9105
Label: CREDITCARDNUMBER
Entity: '9979132607620762'
Context: 'anges to their billing method 9979132607620762147. Assistance needed.'
Entity span: (71, 87)

Record: 9105
Label: CREDITCARDCVV
Entity: '147'
Context: 'illing method 9979132607620762147. Assistance needed.'
Entity span: (87, 90)

Record: 11287
Label: USERNAME
Entity: 'Wilson.OReilly'
Context: 'y 66.94.126.58

In [84]:
# Inspect token labels around the boundary cases

for record_index in sorted(set(e["record_index"] for e in boundary_errors)):
    row = scoped_data[record_index]
    text = row["source_text"]
    entities = row["privacy_mask"]

    raw = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )

    token_ids = raw["input_ids"]
    offsets = raw["offset_mapping"]

    labels = create_bio_labels(text, entities, offsets)
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    print("\n" + "=" * 65)
    print("Record:", record_index)

    for token, (start, end), label in zip(tokens, offsets, labels):
        if start == end:
            continue

        near_error = any(
            err["record_index"] == record_index
            and start < next(
                e["end"] for e in entities
                if e["label"] == err["label"]
                and text[e["start"]:e["end"]] == err["value"]
            ) + 5
            and end > next(
                e["start"] for e in entities
                if e["label"] == err["label"]
                and text[e["start"]:e["end"]] == err["value"]
            ) - 5
            for err in boundary_errors
        )

        if near_error:
            print(f"{token:18} {start:4}-{end:<4} {label}")


Record: 1901
with                 76-80   O
823                  81-84   B-CREDITCARDNUMBER
##53                 84-86   I-CREDITCARDNUMBER
##6                  86-87   I-CREDITCARDNUMBER
##7                  87-88   I-CREDITCARDNUMBER
##3                  88-89   I-CREDITCARDNUMBER
##9                  89-90   I-CREDITCARDNUMBER
##25                 90-92   I-CREDITCARDNUMBER
##38                 92-94   I-CREDITCARDNUMBER
##51                 94-96   I-CREDITCARDNUMBER
##50                 96-98   I-CREDITCARDNUMBER
##26                 98-100  I-CREDITCARDCVV
as                  101-103  O
old                 104-107  O

Record: 2666
sharing             189-196  O
d                   197-198  B-PASSWORD
##z                 198-199  I-PASSWORD
##j                 199-200  I-PASSWORD
##C                 200-201  I-PASSWORD
##X                 201-202  I-PASSWORD
##7                 202-203  I-PASSWORD
##N                 203-204  I-PASSWORD
##um                204-206  I-PASSWORD
##w

In [85]:
for idx in [1901, 2666, 6452, 9105, 11287, 11404, 16604, 17335, 28868]:
    row = scoped_data[idx]
    text = row["source_text"]

    print(f"\n--- Record {idx} ---")

    for e in sorted(row["privacy_mask"], key=lambda x: x["start"]):
        if e["label"] in target_set:
            print(
                e["label"],
                repr(text[e["start"]:e["end"]]),
                (e["start"], e["end"])
            )


--- Record 1901 ---
CREDITCARDNUMBER '8235367392538515' (81, 97)
CREDITCARDCVV '026' (97, 100)

--- Record 2666 ---
IBAN 'MD984KL5166I63568P72550I' (98, 122)
PASSWORD 'dzjCX7Numw7b' (197, 209)

--- Record 6452 ---
IPV4 '30.32.68.100' (75, 87)

--- Record 9105 ---
CREDITCARDNUMBER '9979132607620762' (71, 87)
CREDITCARDCVV '147' (87, 90)

--- Record 11287 ---
IPV4 '66.94.126.58' (101, 113)
USERNAME 'Wilson.OReilly' (129, 143)

--- Record 11404 ---
FIRSTNAME 'Paxton' (24, 30)
LASTNAME 'Treutel' (69, 76)

--- Record 16604 ---
FIRSTNAME 'Ruthie' (90, 96)

--- Record 17335 ---
SSN '756.2177.3265' (238, 251)
SSN '335 15 9744' (251, 262)

--- Record 28868 ---
CREDITCARDNUMBER '4760965854172367' (100, 116)
CREDITCARDCVV '148' (116, 119)


In [86]:
# Count tokens that fully fit inside one target entity
# versus tokens that cross entity boundaries.

full_entity_tokens = 0
crossing_tokens = 0
target_overlap_tokens = 0

for idx in range(len(scoped_data)):
    row = scoped_data[idx]
    text = row["source_text"]
    entities = [
        e for e in row["privacy_mask"]
        if e["label"] in target_set
    ]

    raw = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )

    for start, end in raw["offset_mapping"]:
        if start == end:
            continue

        overlapping = [
            e for e in entities
            if start < e["end"] and end > e["start"]
        ]

        if not overlapping:
            continue

        target_overlap_tokens += 1

        fully_inside_one = any(
            start >= e["start"] and end <= e["end"]
            for e in overlapping
        )

        if fully_inside_one and len(overlapping) == 1:
            full_entity_tokens += 1
        else:
            crossing_tokens += 1

print("Tokens overlapping target entities:", target_overlap_tokens)
print("Tokens fully inside one entity:", full_entity_tokens)
print("Tokens crossing boundaries or entities:", crossing_tokens)

Tokens overlapping target entities: 237182
Tokens fully inside one entity: 237173
Tokens crossing boundaries or entities: 9


In [87]:
# Collect tokens that cross target-entity boundaries

crossing_cases = []

for idx in range(len(scoped_data)):
    row = scoped_data[idx]
    text = row["source_text"]
    entities = [
        e for e in row["privacy_mask"]
        if e["label"] in target_set
    ]

    raw = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )

    for start, end in raw["offset_mapping"]:
        if start == end:
            continue

        overlapping = [
            e for e in entities
            if start < e["end"] and end > e["start"]
        ]

        fully_inside_one = any(
            start >= e["start"] and end <= e["end"]
            for e in overlapping
        )

        if overlapping and not (
            fully_inside_one and len(overlapping) == 1
        ):
            crossing_cases.append({
                "record_index": idx,
                "token": text[start:end],
                "token_span": (start, end),
                "entities": [
                    {
                        "label": e["label"],
                        "value": text[e["start"]:e["end"]],
                        "span": (e["start"], e["end"]),
                    }
                    for e in overlapping
                ],
            })

print("Crossing cases:", len(crossing_cases))

for case in crossing_cases:
    print("\n", case)

Crossing cases: 9

 {'record_index': 1901, 'token': '50', 'token_span': (96, 98), 'entities': [{'label': 'CREDITCARDNUMBER', 'value': '8235367392538515', 'span': (81, 97)}, {'label': 'CREDITCARDCVV', 'value': '026', 'span': (97, 100)}]}

 {'record_index': 2666, 'token': 'bs', 'token_span': (208, 210), 'entities': [{'label': 'PASSWORD', 'value': 'dzjCX7Numw7b', 'span': (197, 209)}]}

 {'record_index': 6452, 'token': '1007', 'token_span': (84, 88), 'entities': [{'label': 'IPV4', 'value': '30.32.68.100', 'span': (75, 87)}]}

 {'record_index': 9105, 'token': '21', 'token_span': (86, 88), 'entities': [{'label': 'CREDITCARDNUMBER', 'value': '9979132607620762', 'span': (71, 87)}, {'label': 'CREDITCARDCVV', 'value': '147', 'span': (87, 90)}]}

 {'record_index': 11287, 'token': 'lys', 'token_span': (141, 144), 'entities': [{'label': 'USERNAME', 'value': 'Wilson.OReilly', 'span': (129, 143)}]}

 {'record_index': 11404, 'token': 'ls', 'token_span': (75, 77), 'entities': [{'label': 'LASTNAME', 'va

In [88]:
def create_bio_labels_v2(text, privacy_mask, offsets):
    labels = []

    target_entities = [
        entity for entity in privacy_mask
        if entity["label"] in target_set
    ]

    for start, end in offsets:
        # Special tokens have offset (0, 0)
        if start == end:
            labels.append(IGNORE_INDEX)
            continue

        overlapping = [
            entity for entity in target_entities
            if start < entity["end"] and end > entity["start"]
        ]

        # No target entity overlaps this token
        if not overlapping:
            labels.append("O")
            continue

        # Ignore tokens that overlap multiple entities
        if len(overlapping) > 1:
            labels.append(IGNORE_INDEX)
            continue

        entity = overlapping[0]

        # Ignore tokens that cross entity boundaries
        if start < entity["start"] or end > entity["end"]:
            labels.append(IGNORE_INDEX)
            continue

        # Assign BIO label
        if start == entity["start"]:
            labels.append(f"B-{entity['label']}")
        else:
            labels.append(f"I-{entity['label']}")

    return labels


print("New BIO labeling function created.")

New BIO labeling function created.


In [89]:
for idx in sorted(set(c["record_index"] for c in crossing_cases)):
    row = scoped_data[idx]

    text = row["source_text"]
    entities = row["privacy_mask"]

    raw = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )

    tokens = tokenizer.convert_ids_to_tokens(raw["input_ids"])

    labels = create_bio_labels_v2(
        text,
        entities,
        raw["offset_mapping"],
    )

    print(f"\n--- Record {idx} ---")

    for token, offset, label in zip(
        tokens, raw["offset_mapping"], labels
    ):
        if label == IGNORE_INDEX:
            start, end = offset
            if start < end and any(
                c["record_index"] == idx
                and start < c["token_span"][1]
                and end > c["token_span"][0]
                for c in crossing_cases
            ):
                print(token, offset, label)


--- Record 1901 ---
##50 (96, 98) -100

--- Record 2666 ---
##bs (208, 210) -100

--- Record 6452 ---
1007 (84, 88) -100

--- Record 9105 ---
##21 (86, 88) -100

--- Record 11287 ---
##lys (141, 144) -100

--- Record 11404 ---
##ls (75, 77) -100

--- Record 16604 ---
[UNK] (90, 98) -100

--- Record 17335 ---
##53 (250, 252) -100

--- Record 28868 ---
##71 (115, 117) -100
